In [20]:
import pandas as pd
import numpy as np
import jieba
import time

from pyecharts.charts import Bar,Line,Map,Page,Pie
from pyecharts import options as opts
from pyecharts.globals import SymbolType
data=pd.read_excel('大众点评.xlsx')
data

,城市,门店名称,地址,品类,营业时间,门店整体评分,口味评分,人均消费,招牌菜
0,上海,BELLOCO倍乐(静安店),新闸路850号(近石门二路),韩国料理,BELLO'CO,4.8,口味: 8.4,人均: 154 元,五花肉:全州拌饭;四色饼;四色饼;奶酪鸡蛋卷;嫩豆腐汤;宫中拌猪排;宫中石锅拌饭;手握饭团;...
1,上海,Thai Gallery泰廊餐厅,南京西路1649号静安公园内,泰国菜,Thai Gallery,4.7,口味: 8.3,人均: 217 元,冬荫功;前餐拼盘;椰汁西米糕;椰汁香茅黄咖...;泰国椰皇;泰廊-冬阴功汤;泰廊特色海鲜.....
2,上海,青鹤谷(虹莘路总店),虹莘路3998号帝宝大厦2楼,韩国料理,"周二至周五 11:30-14:00 17:00-21:00 周六,周日 11:30-15:...",4.8,口味: 8.9,人均: 193 元,五花肉;全州拌饭;四色饼;大酱汤;奶酪鸡蛋卷;嫩豆腐汤;宫中拌猪排;宫中石锅拌饭;手握饭团;...
3,杭州,福缘居酒楼(河坊街店),河坊街35号(驾驶员培训中心旁),浙菜,周一至周日 11:00-14:00 16:30-21:00,4.2,口味: 8.3,人均: 105 元,三鲜汤;上汤菠菜;东坡肉(块);八宝油条;卤鸭;嫩菱芦笋;家常豆腐;干炸带鱼;干炸臭豆腐;望...
4,杭州,朴墅(青芝坞店),玉古路61-1号,浙菜,周一至周日 11:00-13:30 16:30-20:30,4.5,口味: 8.4,人均: 113 元,东洋手撕豆腐;人品大爆发;吱吱鲜芦笋;啫啫紫苏牛蛙;奶油芝士焗南瓜;嫩豌豆;干锅包心菜;年糕...
...,...,...,...,...,...,...,...,...,...
76,郑州,Two Birds One Stone·一石二鸟(木色店),商务内环路和通泰路交叉口永威木色购物公园西门一楼 L1-101,咖啡厅,"周一至周五 09:30-21:30 周六,周日 09:30-22:00",4.2,口味: 8.2,人均: 104 元,SINGHA...;三文鱼塔;三文鱼片牛油...;巴沙炸鱼薯条;意式肉酱面;招牌羊角班尼;水...
77,郑州,出云记(城东路店),城东路与城北路交叉口西50米路南(永利温泉酒店院内东侧),浙菜,周一至周日 11:30-14:30 17:30-21:30,4.5,口味: 8.8,人均: 95 元,出云辣子鸡;双椒扒茼蒿;咸蛋黄茄子;大杏仁炒菠菜;小罐泡馍;手撕包菜;招牌酸梅汤扎;杭州熏鱼...
78,重庆,二火锅(沙滨路总店),沙滨路11号附10号(财信沙滨城市内),重庆火锅,周一至周日 11:00-13:30 16:30-23:00,4.5,口味: 8.9,人均: 115 元,冰汤圆;孜然肉片;毛肚;滑肉;猪黄喉;现炸酥肉;红汤锅底;耗儿鱼;耙牛肉;耙鸡爪;脑花;腰片...
79,重庆,晓小面,洋河四路10号附8号,面馆,周一至周日 全天,4.5,口味: 8.7,人均: 15 元,加肥肠;北冰洋;原汤牛肉;干溜儿宽面;干溜炸酱面;干溜豌杂牛肉;干豌杂;晓小面;杂酱宽面;煎...


In [21]:
#查看数据基本情况
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   城市      81 non-null     object 
 1   门店名称    81 non-null     object 
 2   地址      81 non-null     object 
 3   品类      81 non-null     object 
 4   营业时间    81 non-null     object 
 5   门店整体评分  81 non-null     float64
 6   口味评分    81 non-null     object 
 7   人均消费    81 non-null     object 
 8   招牌菜     78 non-null     object 
dtypes: float64(1), object(8)
memory usage: 5.8+ KB


In [22]:
data.describe()

,门店整体评分
count,81.000000
mean,4.581481
std,0.213372
min,3.800000
25%,4.500000
50%,4.600000
75%,4.700000
max,4.900000


In [23]:
#去除重复值
data.drop_duplicates(inplace=True)
#删除购买人数为空的记录
df_data= data[data['口味评分'].str.contains('口味:')]

# 重置索引
df_data = df_data.reset_index(drop=True)
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   城市      79 non-null     object 
 1   门店名称    79 non-null     object 
 2   地址      79 non-null     object 
 3   品类      79 non-null     object 
 4   营业时间    79 non-null     object 
 5   门店整体评分  79 non-null     float64
 6   口味评分    79 non-null     object 
 7   人均消费    79 non-null     object 
 8   招牌菜     78 non-null     object 
dtypes: float64(1), object(8)
memory usage: 5.7+ KB


In [24]:
df_data

,城市,门店名称,地址,品类,营业时间,门店整体评分,口味评分,人均消费,招牌菜
0,上海,BELLOCO倍乐(静安店),新闸路850号(近石门二路),韩国料理,BELLO'CO,4.8,口味: 8.4,人均: 154 元,五花肉:全州拌饭;四色饼;四色饼;奶酪鸡蛋卷;嫩豆腐汤;宫中拌猪排;宫中石锅拌饭;手握饭团;...
1,上海,Thai Gallery泰廊餐厅,南京西路1649号静安公园内,泰国菜,Thai Gallery,4.7,口味: 8.3,人均: 217 元,冬荫功;前餐拼盘;椰汁西米糕;椰汁香茅黄咖...;泰国椰皇;泰廊-冬阴功汤;泰廊特色海鲜.....
2,上海,青鹤谷(虹莘路总店),虹莘路3998号帝宝大厦2楼,韩国料理,"周二至周五 11:30-14:00 17:00-21:00 周六,周日 11:30-15:...",4.8,口味: 8.9,人均: 193 元,五花肉;全州拌饭;四色饼;大酱汤;奶酪鸡蛋卷;嫩豆腐汤;宫中拌猪排;宫中石锅拌饭;手握饭团;...
3,杭州,福缘居酒楼(河坊街店),河坊街35号(驾驶员培训中心旁),浙菜,周一至周日 11:00-14:00 16:30-21:00,4.2,口味: 8.3,人均: 105 元,三鲜汤;上汤菠菜;东坡肉(块);八宝油条;卤鸭;嫩菱芦笋;家常豆腐;干炸带鱼;干炸臭豆腐;望...
4,杭州,朴墅(青芝坞店),玉古路61-1号,浙菜,周一至周日 11:00-13:30 16:30-20:30,4.5,口味: 8.4,人均: 113 元,东洋手撕豆腐;人品大爆发;吱吱鲜芦笋;啫啫紫苏牛蛙;奶油芝士焗南瓜;嫩豌豆;干锅包心菜;年糕...
...,...,...,...,...,...,...,...,...,...
74,郑州,Two Birds One Stone·一石二鸟(木色店),商务内环路和通泰路交叉口永威木色购物公园西门一楼 L1-101,咖啡厅,"周一至周五 09:30-21:30 周六,周日 09:30-22:00",4.2,口味: 8.2,人均: 104 元,SINGHA...;三文鱼塔;三文鱼片牛油...;巴沙炸鱼薯条;意式肉酱面;招牌羊角班尼;水...
75,郑州,出云记(城东路店),城东路与城北路交叉口西50米路南(永利温泉酒店院内东侧),浙菜,周一至周日 11:30-14:30 17:30-21:30,4.5,口味: 8.8,人均: 95 元,出云辣子鸡;双椒扒茼蒿;咸蛋黄茄子;大杏仁炒菠菜;小罐泡馍;手撕包菜;招牌酸梅汤扎;杭州熏鱼...
76,重庆,二火锅(沙滨路总店),沙滨路11号附10号(财信沙滨城市内),重庆火锅,周一至周日 11:00-13:30 16:30-23:00,4.5,口味: 8.9,人均: 115 元,冰汤圆;孜然肉片;毛肚;滑肉;猪黄喉;现炸酥肉;红汤锅底;耗儿鱼;耙牛肉;耙鸡爪;脑花;腰片...
77,重庆,晓小面,洋河四路10号附8号,面馆,周一至周日 全天,4.5,口味: 8.7,人均: 15 元,加肥肠;北冰洋;原汤牛肉;干溜儿宽面;干溜炸酱面;干溜豌杂牛肉;干豌杂;晓小面;杂酱宽面;煎...


In [25]:
df_data['口味评分'] = df_data['口味评分'].str.extract('(\d+(\.\d+)?)')[0]
df_data['口味评分']

0     8.4
1     8.3
2     8.9
3     8.3
4     8.4
     ... 
74    8.2
75    8.8
76    8.9
77    8.7
78    8.7
Name: 口味评分, Length: 79, dtype: object

In [26]:
# 口味评分数据处理
# 提取评分并转换为数值类型
df_data['口味评分'] = pd.to_numeric(df_data['口味评分'].str.extract('(\d+(\.\d+)?)')[0],errors='coerce')
df_data['口味评分']

0     8.4
1     8.3
2     8.9
3     8.3
4     8.4
     ... 
74    8.2
75    8.8
76    8.9
77    8.7
78    8.7
Name: 口味评分, Length: 79, dtype: float64

In [27]:
#人均消费处理
df_data['人均消费'] = df_data['人均消费'].str.extract('(\d+)').astype('int')
df_data['口味评分']

0     8.4
1     8.3
2     8.9
3     8.3
4     8.4
     ... 
74    8.2
75    8.8
76    8.9
77    8.7
78    8.7
Name: 口味评分, Length: 79, dtype: float64

In [28]:
shop_top10 = df_data.groupby('城市')['人均消费'].sum().sort_values(ascending=False).head(10)
shop_top10

城市
上海    564
大连    499
长沙    426
广州    410
福州    396
济南    356
杭州    335
苏州    305
武汉    303
北京    300
Name: 人均消费, dtype: int32

In [29]:
from pyecharts.globals import CurrentConfig, NotebookType
CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_LAB
from pyecharts.charts import Bar
from pyecharts import options as opts
#内置主题类型可查看 pyecharts.globals.ThemeType
from pyecharts.globals import ThemeType

In [30]:
#条形图
bar1 = Bar()
bar1.add_xaxis(shop_top10.index.tolist())
bar1.add_yaxis('',shop_top10.values.tolist())
bar1.set_global_opts(title_opts=opts.TitleOpts(title='大众点评美食价格排名Top10城市'),
                     xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=-15)),
                     visualmap_opts=opts.VisualMapOpts(max_=28669)
                    )
bar1.load_javascript()

In [31]:
bar1.render_notebook()

In [32]:
province_num = df_data.groupby('城市')['门店整体评分'].sum().sort_values(ascending=False)

province_num

城市
上海     14.3
深圳     14.2
福州     14.2
合肥     14.1
苏州     14.1
沈阳     14.1
南京     14.0
西安     14.0
常州     14.0
北京     14.0
武汉     14.0
长沙     13.8
天津     13.8
宁波     13.8
石家庄    13.7
无锡     13.6
青岛     13.6
厦门     13.6
广州     13.5
哈尔滨    13.5
郑州     13.5
重庆     13.5
温州     13.4
杭州     13.2
成都     12.9
济南     12.6
大连      4.3
Name: 门店整体评分, dtype: float64

In [33]:
# 处理省份地理数据
def store(e):
    if '西藏' in e:
        return '西藏自治区'
    elif '黑龙江' in e:
        return '黑龙江省'
    elif '新疆' in e:
        return '新疆维吾尔自治区'
    elif '广西' in e:
        return '广西壮族自治区'
    elif '内蒙古' in e:
        return '内蒙古自治区'
    elif '宁夏' in e:
        return '宁夏回族自治区'
    else:
        return e

In [34]:
s

NameError: name 's' is not defined

In [35]:
goods["新省份"]=goods['省份'].apply(stone)
goods

NameError: name 'goods' is not defined

In [36]:
map1 = Map()
map1.add("",[list(z) for z in zip(province_num.index.tolist(),province_num.values.tolist())],maptype='china')
map1.set_global_opts(
    title_opts = opts.TitleOpts(title='大众点评国内各地美食评分分布图'),
    visualmap_opts = opts.VisualMapOpts(max_=172277)
)
map1.load_javascript()

In [37]:
map1.render_notebook()

In [42]:
#使用pandas的cut函数自动分箱
#定义区间边界
bins= [0,50,100,150,200,300,500,float('inf')]
labels = ['50元以下','50-100元','100-150元','150-200元','200-300元','300-500元','500元以上']
df_data['价格区间_cut']=pd.cut(df_data['人均消费'],
                           bins=bins,
                           labels=labels,
                           right=False) #左闭右开
#处理缺失值
df_data['价格区间_cut']=df_data['价格区间_cut'].cat.add_categories(['未知']).fillna('未知')
df_data['价格区间_cut']

0     150-200元
1     200-300元
2     150-200元
3     100-150元
4     100-150元
        ...   
74    100-150元
75     50-100元
76    100-150元
77       50元以下
78       50元以下
Name: 价格区间_cut, Length: 79, dtype: category
Categories (8, object): ['50元以下' < '50-100元' < '100-150元' < '150-200元' < '200-300元' < '300-500元' < '500元以上' < '未知']

In [41]:
s=df_data['价格区间_cut'].value_counts()
s

价格区间_cut
50-100元     40
100-150元    22
50元以下        7
150-200元     7
200-300元     2
300-500元     1
500元以上       0
未知           0
Name: count, dtype: int64

In [44]:
bar3 = Bar()
bar3.add_xaxis(s.index.tolist())
bar3.add_yaxis('',s.values.tolist())
bar3.set_global_opts(title_opts=opts.TitleOpts(title='大众点评人均消费不同价格区间情况'),
                     visualmap_opts=opts.VisualMapOpts(max_=900))
bar3.load_javascript()

In [45]:
bar3.render_notebook()

In [46]:
data_pair = [list(z) for z in zip(s.index.tolist(),s.values.tolist())]

#绘制饼图
pie1 = Pie()
pie1.add('',data_pair, radius=['35%','60%'])
pie1.set_global_opts(title_opts=opts.TitleOpts(title='大众点评不同价格区间人均消费的表现'),
                     legend_opts=opts.LegendOpts(orient='vertical',pos_top='15%',pos_left='2%'))
pie1.set_series_opts(label_opts=opts.LabelOpts(formatter="{b}:{d}%"))
pie1.set_colors(['#EF9050', '#3B7BA9', '#6FB27C', '#FFAF34', '#D8BFD8', '#00BFFF', '#7FFFAA'])
pie1.load_javascript()
pie1.render_notebook()